<a href="https://colab.research.google.com/github/flg37/Beng-Dissertation/blob/main/Model_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
###
## Model 2 Training
###

"""
Multi-Modal State Space Model (Mamba) for Battery State-of-Health (SOH) Estimation
-----------------------------------------------------------------------------------
This script implements a late-fusion multi-modal neural network to predict battery SOH.
It leverages State Space Models (SSMs), specifically Vision Mamba for spatial voltage
maps and standard Mamba for temporal cycling sequences, concatenated with an MLP for
static scalar features. The training loop utilizes a nested matrix evaluation (multiple
data splits crossed with multiple weight initializations) to ensure statistical
significance of the empirical results.
"""
import numpy as np
import torch
import pandas as pd
import pickle
import sys
import random
import os
from torch.utils.data import Dataset, DataLoader
import math
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F
import gc
import json
import datetime
from torch.optim.lr_scheduler import OneCycleLR
import torch.nn as nn
from mamba_ssm import Mamba
# ============================================================================
# 0. DETERMINISTIC EXECUTION (REPRODUCIBILITY)
# ============================================================================
def set_seed(seed=20):
"""
Enforces deterministic execution across Python, NumPy, and PyTorch.
87
Crucial for academic reproducibility and fair architectural comparisons.
"""
random.seed(seed)
os.environ[’PYTHONHASHSEED’] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.backends.cudnn.benchmark = False # Disables auto-tuning for strict determinism
# ============================================================================
# 1. ENVIRONMENT CONFIGURATION & DATA INGESTION
# ============================================================================
# Paths configured for Google Colab/Drive cloud infrastructure
DRIVE_PREFIX = "/content/drive/MyDrive/Beng_Dissertation"
DRIVE_PREFIX2 = "/content/drive/MyDrive"
device = torch.device(’cuda’ if torch.cuda.is_available() else ’cpu’)
INPUT_DATA_PATH = f"{DRIVE_PREFIX}/multimodal_features_synced2.pkl"
LABELS_CSV_PATH = f"{DRIVE_PREFIX2}/session_level_soh_natural_final2.csv"
print(f"Loading extracted multimodal feature tensors from {INPUT_DATA_PATH}...")
try:
with open(INPUT_DATA_PATH, ’rb’) as f:
raw_features = pickle.load(f)
print(f" Loaded {len(raw_features)} sequence blocks.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading features: {e}")
print(f"Loading empirical SOH targets from {LABELS_CSV_PATH}...")
try:
with open(LABELS_CSV_PATH, ’rb’) as f:
labels_df = pd.read_csv(f)
print(f" Loaded {len(labels_df)} target labels.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading labels: {e}")
# ============================================================================
# 2. FEATURE-TARGET SYNCHRONIZATION
# ============================================================================
print(" Performing inner join of latent features with empirical SOH labels...")
# Robust column detection to handle upstream dataset versioning
vid_col = ’vehicle_id_split’ if ’vehicle_id_split’ in labels_df.columns else ’vehicle_id’
labels_df[vid_col] = labels_df[vid_col].astype(str)
labels_df[’session_id’] = labels_df[’session_id’].astype(int)
target_col = ’soh_final’ if ’soh_final’ in labels_df.columns else ’soh’
# Create a hash map for O(1) target label lookups
label_dict = labels_df.set_index([vid_col, ’session_id’])[target_col].to_dict()
clean_features_master = []
missing_labels = 0
# Cross-reference extracted features with available ground-truth labels
for feat in raw_features:
vid = str(feat[’vehicle_id’])
88
sid = int(feat[’session_id’])
if (vid, sid) in label_dict:
feat_copy = feat.copy()
# Scale normalized SOH back to absolute Ampere-hours (Ah) using nominal 155.0 Ah capacity
feat_copy[’measured_soh_ah’] = label_dict[(vid, sid)] * 155.0
clean_features_master.append(feat_copy)
else:
missing_labels += 1
print(f" Successfully synchronized {len(clean_features_master)} complete training samples.")
if missing_labels > 0:
print(f" Dropped {missing_labels} samples due to missing ground-truth labels.")
# ============================================================================
# 3. OPTIMIZED DATASET CLASS & PREPROCESSING
# ============================================================================
class OptimalBatteryDataset(Dataset):
"""
PyTorch Dataset mapping multi-modal battery telemetry to tensor representations.
Includes robust NaN imputation and modality-specific normalization strategies.
"""
def __init__(self, features, point_mean, point_std):
self.features = features
self.point_mean = point_mean
self.point_std = point_std
def __len__(self): return len(self.features)
def __getitem__(self, idx):
try:
feat = self.features[idx]
if feat is None: raise ValueError(f"Feature at index {idx} is None")
# --- Modality A: 2D Spatial Voltage Map ---
# Represents spatial degradation distributions. Normalized via instance Z-score.
v_map = feat[’voltage_map_2d’]
if np.isnan(v_map).any():
v_map = np.nan_to_num(v_map, nan=0.0)
v_map = (v_map - np.mean(v_map)) / (np.std(v_map) + 1e-6)
t_vmap = torch.FloatTensor(v_map).unsqueeze(0) # Shape: [1, H, W]
# --- Modality B: 1D Temporal Sequences ---
# Capacity and temperature dynamics over the charging cycle.
# Statically scaled by 100 to map roughly to a [0, 1] range.
cap = np.nan_to_num(feat[’capacity_seq_1d’] / 100.0, nan=0.0)
temp = np.nan_to_num(feat[’temperature_seq_1d’] / 100.0, nan=0.0)
seq_1d = torch.FloatTensor(np.stack([cap, temp], axis=0)) # Shape: [2, 151]
# --- Modality C: 1D Static Point Features ---
# Extracted session metadata. Normalized using global dataset statistics.
raw_p_feat = feat[’point_features’]
if np.isnan(raw_p_feat).any():
raw_p_feat = np.nan_to_num(raw_p_feat, nan=0.0)
p_feat = (torch.FloatTensor(raw_p_feat) - self.point_mean) / self.point_std
# Target SOH (Re-normalized by nominal capacity for MSE stability)
89
label = torch.FloatTensor([feat[’measured_soh_ah’] / 155.0])
return t_vmap, seq_1d, p_feat, label
except Exception:
# Fallback zero-tensors to prevent dataloader crashes on corrupted shards
return (
torch.zeros((1, 96, 96), dtype=torch.float32),
torch.zeros((2, 151), dtype=torch.float32),
torch.zeros((15,), dtype=torch.float32),
torch.zeros((1,), dtype=torch.float32)
)
def seed_worker(worker_id):
"""Ensures deterministic batch generation across multi-processing dataloaders."""
worker_seed = torch.initial_seed() % 2**32
np.random.seed(worker_seed)
random.seed(worker_seed)
# ==============================================================================
# 4. ARCHITECTURAL DEFINITIONS (MULTI-MODAL STATE-SPACE NETWORK)
# ==============================================================================
class CrossScanMambaBlock(nn.Module):
"""
Implements a 4-way cross-scan mechanism (Vim/Vision Mamba approach).
Standard SSMs (like 1D Mamba) suffer from directional bias and break spatial
locality. By sweeping horizontally and vertically in both forward and reverse
directions, this block captures global 2D spatial context efficiently.
"""
def __init__(self, embed_dim, grid_size):
super().__init__()
self.grid_size = grid_size
self.norm = nn.LayerNorm(embed_dim)
# 4 distinct Mamba blocks for directional scanning
self.mamba_h_fwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_h_bwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_v_fwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_v_bwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.out_proj = nn.Linear(embed_dim * 4, embed_dim)
def forward(self, x):
B, L, D = x.shape
H = W = self.grid_size
x_norm = self.norm(x)
# 1. Horizontal Forward
out_h_fwd = self.mamba_h_fwd(x_norm)
# 2. Horizontal Backward
out_h_bwd = self.mamba_h_bwd(x_norm.flip([1])).flip([1])
# 3. Vertical Forward (Requires grid transposition)
spatial_v = x_norm.view(B, H, W, D).transpose(1, 2).reshape(B, L, D)
out_v_fwd = self.mamba_v_fwd(spatial_v)
out_v_fwd = out_v_fwd.view(B, W, H, D).transpose(1, 2).reshape(B, L, D)
90
# 4. Vertical Backward
spatial_v_bwd = spatial_v.flip([1])
out_v_bwd = self.mamba_v_bwd(spatial_v_bwd).flip([1])
out_v_bwd = out_v_bwd.view(B, W, H, D).transpose(1, 2).reshape(B, L, D)
# Feature fusion and residual connection
merged = torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
out = self.out_proj(merged)
return x + out
class VisionMambaEncoder(nn.Module):
"""
Translates 2D topological voltage maps into latent representations.
Utilizes a lightweight CNN stem to aggressively downsample the spatial resolution,
reducing sequence length and thereby computational complexity (O(N) for SSMs).
"""
def __init__(self, img_size=96, in_chans=1, embed_dim=128, depth=3):
super().__init__()
# Convolutional Stem: 96x96 -> 12x12 (144 tokens). Acts as a feature extractor
# prior to sequence modeling, bridging CNN inductive biases with SSM global context.
self.stem = nn.Sequential(
nn.Conv2d(in_chans, embed_dim // 4, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim // 4),
nn.GELU(),
nn.Conv2d(embed_dim // 4, embed_dim // 2, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim // 2),
nn.GELU(),
nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim),
nn.GELU()
)
self.grid_size = img_size // 8
self.num_patches = self.grid_size ** 2
# Learnable spatial positional encoding
self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
self.blocks = nn.ModuleList([
CrossScanMambaBlock(embed_dim=embed_dim, grid_size=self.grid_size)
for _ in range(depth)
])
self.norm_f = nn.LayerNorm(embed_dim)
def forward(self, x):
x = self.stem(x)
x = x.flatten(2).transpose(1, 2) # [B, 128, 12, 12] -> [B, 144, 128]
x = x + self.pos_embed
for blk in self.blocks:
x = blk(x)
x = self.norm_f(x)
# Global Average Pooling to yield a single fixed-length latent vector
return x.mean(dim=1)
91
class UnifiedMambaSOH(nn.Module):
"""
Late-Fusion Multi-Modal Architecture.
Independently encodes spatial maps (Vision Mamba), temporal sequences
(1D Mamba + Attention), and static metadata (MLP), before concatenating
them into a unified latent space for non-linear regression.
"""
def __init__(self, seq_dim=2, static_dim=15, mamba_dim=128):
super().__init__()
# --- Branch 1: Spatial Topologies ---
self.vim = VisionMambaEncoder(img_size=96, in_chans=1, embed_dim=mamba_dim, depth=3)
# --- Branch 2: Temporal Dynamics ---
# Projects sequence variables + temporal position embedding into SSM dimension
self.seq_proj = nn.Linear(seq_dim + 1, mamba_dim)
self.seq_mamba = Mamba(d_model=mamba_dim, d_state=32, d_conv=4, expand=4)
# Attention-based temporal pooling (learns which time steps indicate degradation)
self.seq_attention = nn.Sequential(
nn.Linear(mamba_dim, mamba_dim // 2),
nn.Tanh(),
nn.Linear(mamba_dim // 2, 1),
nn.Softmax(dim=1)
)
# --- Branch 3: Static Point Features ---
point_out_dim = 64
self.point_encoder = nn.Sequential(
nn.Linear(static_dim, 128),
nn.LayerNorm(128),
nn.GELU(),
nn.Dropout(0.06),
nn.Linear(128, point_out_dim)
)
# --- Modality Fusion & Regressor Head ---
self.norm_vim = nn.LayerNorm(mamba_dim)
self.norm_seq = nn.LayerNorm(mamba_dim)
self.norm_point = nn.LayerNorm(point_out_dim)
fusion_dim = mamba_dim + mamba_dim + point_out_dim
self.fusion_norm = nn.LayerNorm(fusion_dim)
self.regressor = nn.Sequential(
nn.Linear(fusion_dim, 256),
nn.LayerNorm(256),
nn.GELU(),
nn.Dropout(0.02),
nn.Linear(256, 64),
nn.GELU(),
nn.Dropout(0.02)
nn.Linear(64, 1) # Predicts normalized scalar SOH
)
def forward(self, v_map, s_1d, p_feat):
92
# 1. Spatial embedding
vim_out = self.norm_vim(self.vim(v_map))
# 2. Temporal embedding with relative time pos-encoding
s_1d = s_1d.transpose(1, 2)
batch_size, seq_len, _ = s_1d.shape
time_pos = torch.linspace(0, 1, steps=seq_len, device=s_1d.device)
time_pos = time_pos.view(1, seq_len, 1).expand(batch_size, seq_len, 1)
s_1d = torch.cat([s_1d, time_pos], dim=2)
m_out = self.seq_mamba(self.seq_proj(s_1d))
attn_weights = self.seq_attention(m_out)
seq_out = self.norm_seq(torch.sum(m_out * attn_weights, dim=1))
# 3. Static embedding
p_encoded = self.norm_point(self.point_encoder(p_feat))
# 4. Concatenation and Regression
combined = self.fusion_norm(torch.cat([vim_out, seq_out, p_encoded], dim=1))
if self.training:
combined = nn.functional.dropout(combined, p=0.02)
return self.regressor(combined)
criterion = nn.MSELoss()
# ==============================================================================
# 5. ENSEMBLE TRAINING & CROSS-VALIDATION MATRIX
# ==============================================================================
# Robust evaluation matrix: Evaluates 5 data permutations (via master seeds)
# against 5 weight initializations (via run suffixes), totaling 25 distinct models.
model_seeds = [44, 400, 300, 30, 20]
run_suffixes = [’a’, ’b’, ’c’, ’d’, ’e’]
epochs = 100
accumulation_steps = 1
BATCH_SIZE = 128
all_seed_results = {}
for current_seed in model_seeds:
print(f"\n{’=’*70}\n STARTING MASTER DATA SPLIT SEED: {current_seed}\n{’=’*70}")
# Induce specific shuffle permutation for Walk-Forward or K-Fold equivalent evaluation
set_seed(current_seed)
features_for_this_seed = list(clean_features_master)
random.shuffle(features_for_this_seed)
# Standard 80/20 Train-Test Split based on shuffled index
split_idx = int(0.8 * len(features_for_this_seed))
train_features = features_for_this_seed[:split_idx]
test_features = features_for_this_seed[split_idx:]
# Calculate global dataset statistics *only* on training set to prevent data leakage
train_points = np.stack([np.nan_to_num(f[’point_features’], nan=0.0) for f in train_features])
point_mean = torch.FloatTensor(train_points.mean(axis=0))
point_std = torch.FloatTensor(train_points.std(axis=0))
93
point_std[point_std == 0] = 1.0 # Prevent division by zero
ensemble_mapes = []
# Inner loop: Varying model weight initializations for ensemble robustness
for idx, suffix in enumerate(run_suffixes):
best_model_path = f"{DRIVE_PREFIX}/Mamba2/best_Unified_{current_seed}_{suffix}.pth"
print(f"\n{’-’*60}\n ENSEMBLE RUN ’{suffix.upper()}’ FOR MASTER SEED {current_seed}\n{’-’*60# Unique run seed ensures varying initial weights while maintaining reproducibility
run_seed = current_seed + (idx * 1000)
set_seed(run_seed)
train_dataset = OptimalBatteryDataset(train_features, point_mean, point_std)
test_dataset = OptimalBatteryDataset(test_features, point_mean, point_std)
# DataLoader configuration with explicit generator for deterministic batching
g = torch.Generator()
g.manual_seed(run_seed)
train_loader = DataLoader(
train_dataset, batch_size=BATCH_SIZE, shuffle=True,
drop_last=True, worker_init_fn=seed_worker, generator=g
)
test_loader = DataLoader(
test_dataset, batch_size=BATCH_SIZE, shuffle=False
)
model = UnifiedMambaSOH().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
# OneCycleLR policy: Allows super-convergence and acts as a form of regularization
scheduler = OneCycleLR(
optimizer, max_lr=0.001, steps_per_epoch=len(train_loader),
epochs=epochs, pct_start=0.35, div_factor=25.0
)
best_mape = float(’inf’)
patience = 55
epochs_no_improve = 0
# --- EPOCH LOOP ---
for epoch in range(1, epochs + 1):
model.train()
train_mse_sum, train_mape_sum = 0.0, 0.0
valid_train_batches = 0
optimizer.zero_grad()
for i, (v_map, s_1d, p_feat, lbl) in enumerate(train_loader):
v_map, s_1d, p_feat, lbl = v_map.to(device), s_1d.to(device), p_feat.to(device), lbl# Data Augmentation: Stochastic noise injection on temporal features to prevent overfif model.training:
noise = torch.randn_like(s_1d) * 0.001
s_1d = s_1d + noise
94
lbl = lbl.view(-1, 1)
out_norm = model(v_map, s_1d, p_feat).view(-1, 1)
loss = criterion(out_norm, lbl) / accumulation_steps
# Safeguard against vanishing/exploding gradients typical in long temporal processingif torch.isnan(loss) or torch.isinf(loss): continue
loss.backward()
if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Gradient clipping
optimizer.step()
optimizer.zero_grad()
scheduler.step()
# Calculate metrics in absolute units (Ampere-hours) for physical interpretation
with torch.no_grad():
out_real = out_norm * 155.0
lbl_real = lbl * 155.0
train_mse_sum += torch.sum((out_real - lbl_real) ** 2).item()
train_mape_sum += (torch.sum(torch.abs(out_real - lbl_real) / (lbl_real + 1e-6))valid_train_batches += lbl.size(0)
# --- VALIDATION LOOP ---
model.eval()
val_mse_sum, val_mape_sum = 0.0, 0.0
valid_val_batches = 0
with torch.no_grad():
for v_map, s_1d, p_feat, lbl in test_loader:
v_map, s_1d, p_feat, lbl = v_map.to(device), s_1d.to(device), p_feat.to(device),lbl = lbl.view(-1, 1)
out_norm = model(v_map, s_1d, p_feat).view(-1, 1)
out_real, lbl_real = out_norm * 155.0, lbl * 155.0
sq_err = torch.sum((out_real - lbl_real) ** 2)
abs_pct = torch.sum(torch.abs(out_real - lbl_real) / (lbl_real + 1e-6)) * 100.0
if not torch.isnan(sq_err):
val_mse_sum += sq_err.item()
val_mape_sum += abs_pct.item()
valid_val_batches += lbl.size(0)
avg_train_rmse = np.sqrt(train_mse_sum / valid_train_batches) if valid_train_batches > 0avg_train_mape = train_mape_sum / valid_train_batches if valid_train_batches > 0 else 0.0avg_val_rmse = np.sqrt(val_mse_sum / valid_val_batches) if valid_val_batches > 0 else 999avg_val_mape = val_mape_sum / valid_val_batches if valid_val_batches > 0 else 999.0
# Model Checkpointing based on Mean Absolute Percentage Error (MAPE)
if avg_val_mape < best_mape:
best_mape = avg_val_mape
epochs_no_improve = 0
try:
torch.save(model.state_dict(), best_model_path)
saved_msg = "Saved Best"
95
except Exception as e:
saved_msg = "Save failed"
else:
epochs_no_improve += 1
saved_msg = f"No imp ({epochs_no_improve}/{patience})"
current_lr = scheduler.get_last_lr()[0]
print(f"Epoch [{epoch}/{epochs}] | LR: {current_lr:.6f} | "
f"Train RMSE: {avg_train_rmse:.4f} Ah MAPE: {avg_train_mape:.2f}% | "
f"Val RMSE: {avg_val_rmse:.4f} Ah MAPE: {avg_val_mape:.2f}% | {saved_msg}")
# Early Stopping Protocol
if epochs_no_improve >= patience:
print(f"Early Stopping triggered on Epoch {epoch}")
break
print(f"Seed {current_seed}, Run {suffix.upper()} Complete. Best Val MAPE: {best_mape:.2f}%")ensemble_mapes.append(best_mape)
# Stringent VRAM cleanup to prevent memory leaks during 25x nested iteration
del model, optimizer, scheduler, train_loader, test_loader, train_dataset, test_dataset
gc.collect()
torch.cuda.empty_cache()
avg_seed_mape = np.mean(ensemble_mapes)
all_seed_results[current_seed] = {
"individual_runs": ensemble_mapes,
"ensemble_average": avg_seed_mape
}
print(f"AVERAGE ENSEMBLE MAPE FOR SPLIT SEED {current_seed}: {avg_seed_mape:.2f}%")
# ==============================================================================
# 6. RESULTS SERIALIZATION
# ==============================================================================
# Export robust evaluation metrics for report generation and plotting
results_dict = {
"timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
"model_type": "UnifiedSOH_Multimodal",
"matrix_results": all_seed_results
}
results_file_path = f"{DRIVE_PREFIX}/Mamba2/25_run2_ensemble_results.json"
with open(results_file_path, "w") as f:
json.dump(results_dict, f, indent=4)
print(f"\n All 25 ensemble iterations complete! Statistical matrix saved to: {results_file_path}")

In [ ]:
###
## Model 2 Evaluation
###

"""
Evaluation script for a unified multimodal Mamba-based battery SOH estimator.
This script:
1. Loads pre-extracted multimodal feature blocks and session-level SOH labels.
2. Synchronizes features with valid labels and constructs a PyTorch Dataset.
3. Defines a unified state-space architecture consisting of:
- a Vision Mamba encoder for 2D voltage maps,
96
- a temporal Mamba encoder for 1D sequential signals,
- an MLP encoder for static point features.
4. Evaluates 25 trained models (5 seeds × 5 runs).
5. Aggregates evaluation metrics into an academic-style summary table.
6. Generates publication-ready plots for regression quality, model stability,
degradation trajectories, and temporal attention interpretability.
"""
import torch
import numpy as np
import pandas as pd
import pickle
import sys
import random
import os
import math
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from mamba_ssm import Mamba
# Plotting libraries for publication-quality figures
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
# ============================================================================
# 0. REPRODUCIBILITY: GLOBAL RANDOM SEED
# ============================================================================
def set_seed(seed: int) -> None:
"""
Set random seeds for reproducible evaluation.
This function fixes the random state for Python, NumPy, and PyTorch, and
configures cuDNN for deterministic behaviour. This is necessary because
the evaluation reconstructs the train/test split for each seed and must
remain consistent across runs.
Args:
seed (int): Random seed used for all stochastic operations.
"""
random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# ============================================================================
# 1. GOOGLE DRIVE CONFIGURATION AND DATA LOADING
# ============================================================================
# Base folders for Google Drive in Colab
DRIVE_PREFIX = "/content/drive/MyDrive/Final_Year_Project"
DRIVE_PREFIX2 = "/content/drive/MyDrive"
97
# Input feature file and session-level SOH label file
INPUT_DATA_PATH = f"{DRIVE_PREFIX}/multimodal_features_synced4.pkl"
LABELS_CSV_PATH = f"{DRIVE_PREFIX2}/session_level_soh_natural_final1.csv"
# Use GPU if available; otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Loading extracted features from {INPUT_DATA_PATH}...")
try:
# raw_features is a list of multimodal session dictionaries
with open(INPUT_DATA_PATH, "rb") as f:
raw_features = pickle.load(f)
print(f" Loaded {len(raw_features)} feature blocks.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading features: {e}")
print(f" Loading SOH labels from {LABELS_CSV_PATH}...")
try:
# labels_df contains session-level SOH targets
with open(LABELS_CSV_PATH, "rb") as f:
labels_df = pd.read_csv(f)
print(f" Loaded {len(labels_df)} target SOH labels.")
except Exception as e:
sys.exit(f"\n CRITICAL ERROR loading labels: {e}")
# ============================================================================
# 2. SYNCHRONIZE FEATURES WITH LABELS
# ============================================================================
print(" Merging features with target labels...")
# Handle possible alternative vehicle identifier column names
vid_col = "vehicle_id_split" if "vehicle_id_split" in labels_df.columns else "vehicle_id"
labels_df[vid_col] = labels_df[vid_col].astype(str)
labels_df["session_id"] = labels_df["session_id"].astype(int)
# Prefer smoothed final SOH if available; otherwise fall back to raw SOH
target_col = "soh_final" if "soh_final" in labels_df.columns else "soh"
label_dict = labels_df.set_index([vid_col, "session_id"])[target_col].to_dict()
clean_features = []
for feat in raw_features:
vid = str(feat["vehicle_id"])
sid = int(feat["session_id"])
# Keep only feature blocks with a valid target label
if (vid, sid) in label_dict:
feat_copy = feat.copy()
# Convert SOH fraction to absolute capacity in Ah using nominal capacity
feat_copy["measured_soh_ah"] = label_dict[(vid, sid)] * 155.0
clean_features.append(feat_copy)
# ============================================================================
# 3. DATASET CLASS AND MODEL ARCHITECTURE
# ============================================================================
class OptimalBatteryDataset(Dataset):
98
"""
PyTorch dataset for multimodal battery SOH estimation.
Each sample contains:
- voltage_map_2d: 2D voltage-map representation.
- capacity_seq_1d: session-level capacity trajectory.
- temperature_seq_1d: session-level temperature trajectory.
- point_features: engineered scalar health indicators.
- measured_soh_ah: absolute capacity target in Ah.
The dataset performs:
1. Per-sample z-score normalization of the 2D voltage map.
2. Scaling of sequential signals by 1/100 for numerical stability.
3. Train-set normalization of static point features.
4. Conversion of capacity targets back to normalized SOH in [0, 1].
"""
def __init__(self, features, point_mean: torch.Tensor, point_std: torch.Tensor):
"""
Args:
features (list[dict]): List of multimodal feature dictionaries.
point_mean (torch.Tensor): Train-set mean of point_features.
point_std (torch.Tensor): Train-set std of point_features.
"""
self.features = features
self.point_mean = point_mean
self.point_std = point_std
def __len__(self) -> int:
return len(self.features)
def __getitem__(self, idx: int):
"""
Construct a single multimodal sample.
Returns:
t_vmap (torch.FloatTensor): Voltage map input, shape [1, H, W].
seq_1d (torch.FloatTensor): Sequential input, shape [2, T].
p_feat (torch.FloatTensor): Normalized static feature vector, shape [F].
label (torch.FloatTensor): Normalized SOH target, shape [1].
"""
try:
feat = self.features[idx]
# --- 2D voltage map preprocessing ---
v_map = feat["voltage_map_2d"]
if np.isnan(v_map).any():
v_map = np.nan_to_num(v_map, nan=0.0)
# Per-sample standardization
v_map = (v_map - np.mean(v_map)) / (np.std(v_map) + 1e-6)
t_vmap = torch.FloatTensor(v_map).unsqueeze(0)
# --- 1D sequential features: capacity and temperature ---
cap = np.nan_to_num(feat["capacity_seq_1d"] / 100.0, nan=0.0)
temp = np.nan_to_num(feat["temperature_seq_1d"] / 100.0, nan=0.0)
seq_1d = torch.FloatTensor(np.stack([cap, temp], axis=0))
# --- Static engineered point features ---
99
raw_p_feat = feat["point_features"]
if np.isnan(raw_p_feat).any():
raw_p_feat = np.nan_to_num(raw_p_feat, nan=0.0)
p_feat = (torch.FloatTensor(raw_p_feat) - self.point_mean) / self.point_std
# --- Target label: normalized SOH ---
label = torch.FloatTensor([feat["measured_soh_ah"] / 155.0])
return t_vmap, seq_1d, p_feat, label
except Exception:
# Safe fallback to preserve tensor shapes if a corrupted sample is encountered
return (
torch.zeros((1, 96, 96), dtype=torch.float32),
torch.zeros((2, 151), dtype=torch.float32),
torch.zeros((15,), dtype=torch.float32),
torch.zeros((1,), dtype=torch.float32),
)
class CrossScanMambaBlock(nn.Module):
"""
Cross-directional Vision Mamba block for spatial token processing.
The block processes the 2D token grid in four directions:
1. Horizontal forward,
2. Horizontal backward,
3. Vertical forward,
4. Vertical backward.
The outputs from all four scans are concatenated and linearly projected
back to the embedding dimension, followed by a residual connection.
"""
def __init__(self, embed_dim: int, grid_size: int):
super().__init__()
self.grid_size = grid_size
self.norm = nn.LayerNorm(embed_dim)
self.mamba_h_fwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_h_bwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_v_fwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
self.mamba_v_bwd = Mamba(d_model=embed_dim, d_state=16, d_conv=4, expand=2)
# Projection used to fuse four directional scans into a single representation
self.out_proj = nn.Linear(embed_dim * 4, embed_dim)
def forward(self, x: torch.Tensor) -> torch.Tensor:
"""
Args:
x (torch.Tensor): Token sequence of shape [B, L, D].
Returns:
torch.Tensor: Updated token sequence of shape [B, L, D].
"""
B, L, D = x.shape
H = W = self.grid_size
x_norm = self.norm(x)
100
# 1. Horizontal forward scan
out_h_fwd = self.mamba_h_fwd(x_norm)
# 2. Horizontal backward scan
out_h_bwd = self.mamba_h_bwd(x_norm.flip([1])).flip([1])
# 3. Vertical forward scan: transpose the token grid before scanning
spatial_v = x_norm.view(B, H, W, D).transpose(1, 2).reshape(B, L, D)
out_v_fwd = self.mamba_v_fwd(spatial_v)
out_v_fwd = out_v_fwd.view(B, W, H, D).transpose(1, 2).reshape(B, L, D)
# 4. Vertical backward scan
spatial_v_bwd = spatial_v.flip([1])
out_v_bwd = self.mamba_v_bwd(spatial_v_bwd).flip([1])
out_v_bwd = out_v_bwd.view(B, W, H, D).transpose(1, 2).reshape(B, L, D)
# Concatenate the four directional outputs and project back to embed_dim
merged = torch.cat([out_h_fwd, out_h_bwd, out_v_fwd, out_v_bwd], dim=-1)
out = self.out_proj(merged)
# Residual connection stabilizes optimization and preserves input information
return x + out
class VisionMambaEncoder(nn.Module):
"""
Vision Mamba encoder for the 2D voltage-map modality.
Architecture:
- 3-layer convolutional stem reduces the 96×96 image to a 12×12 token grid.
- Learnable positional embeddings are added to the flattened token sequence.
- Several CrossScanMamba blocks process spatial information bidirectionally.
- Global average pooling across tokens produces the final spatial embedding.
"""
def __init__(
self,
img_size: int = 96,
in_chans: int = 1,
embed_dim: int = 128,
depth: int = 3,
):
super().__init__()
# Convolutional stem: progressively downsamples 96×96 to 12×12
self.stem = nn.Sequential(
nn.Conv2d(in_chans, embed_dim // 4, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim // 4),
nn.GELU(),
nn.Conv2d(embed_dim // 4, embed_dim // 2, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim // 2),
nn.GELU(),
nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
nn.BatchNorm2d(embed_dim),
nn.GELU(),
)
101
# 96 / 8 = 12, so the final grid is 12×12 = 144 spatial tokens
self.grid_size = img_size // 8
self.num_patches = self.grid_size ** 2
# Learnable positional embeddings preserve spatial ordering information
self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim))
# Stack of cross-scan Mamba blocks for spatial token modeling
self.blocks = nn.ModuleList(
[
CrossScanMambaBlock(embed_dim=embed_dim, grid_size=self.grid_size)
for _ in range(depth)
]
)
self.norm_f = nn.LayerNorm(embed_dim)
def forward(self, x: torch.Tensor) -> torch.Tensor:
"""
Args:
x (torch.Tensor): Input voltage map, shape [B, 1, 96, 96].
Returns:
torch.Tensor: Spatial embedding of shape [B, embed_dim].
"""
# 1. Extract low-level spatial features with the convolutional stem
x = self.stem(x)
# 2. Flatten spatial grid into a token sequence: [B, C, H, W] -> [B, L, D]
x = x.flatten(2).transpose(1, 2)
# 3. Add learnable positional embeddings
x = x + self.pos_embed
# 4. Process tokens with cross-scan Vision Mamba blocks
for blk in self.blocks:
x = blk(x)
x = self.norm_f(x)
# 5. Global average pooling across tokens
return x.mean(dim=1)
class UnifiedMambaSOH(nn.Module):
"""
Unified multimodal state-space model for battery SOH regression.
Branches:
- Spatial branch: Vision Mamba encoder for 2D voltage maps.
- Temporal branch: standard Mamba encoder with attention pooling for
sequential capacity-temperature trajectories.
- Static branch: MLP encoder for engineered point features.
The three branch outputs are normalized, concatenated, and passed through
a fusion MLP to predict normalized SOH.
"""
102
def __init__(self, seq_dim: int = 2, static_dim: int = 15, mamba_dim: int = 128):
super().__init__()
# 1. Spatial encoder: Vision Mamba for voltage maps
self.vim = VisionMambaEncoder(
img_size=96, in_chans=1, embed_dim=mamba_dim, depth=3
)
# 2. Temporal encoder: Mamba over sequential capacity-temperature signals
self.seq_proj = nn.Linear(seq_dim + 1, mamba_dim)
self.seq_mamba = Mamba(d_model=mamba_dim, d_state=32, d_conv=4, expand=4)
self.seq_attention = nn.Sequential(
nn.Linear(mamba_dim, mamba_dim // 2),
nn.Tanh(),
nn.Linear(mamba_dim // 2, 1),
nn.Softmax(dim=1),
)
# 3. Static encoder: MLP for engineered point features
point_out_dim = 64
self.point_encoder = nn.Sequential(
nn.Linear(static_dim, 128),
nn.LayerNorm(128),
nn.GELU(),
nn.Dropout(0.06),
nn.Linear(128, point_out_dim),
)
# 4. Fusion block
self.norm_vim = nn.LayerNorm(mamba_dim)
self.norm_seq = nn.LayerNorm(mamba_dim)
self.norm_point = nn.LayerNorm(point_out_dim)
fusion_dim = mamba_dim + mamba_dim + point_out_dim
self.fusion_norm = nn.LayerNorm(fusion_dim)
self.regressor = nn.Sequential(
nn.Linear(fusion_dim, 256),
nn.LayerNorm(256),
nn.GELU(),
nn.Dropout(0.02),
nn.Linear(256, 64),
nn.GELU(),
nn.Dropout(0.02),
nn.Linear(64, 1),
)
def forward(
self,
v_map: torch.Tensor,
s_1d: torch.Tensor,
p_feat: torch.Tensor,
return_attention: bool = False,
):
"""
Forward pass for unified multimodal SOH prediction.
Args:
103
v_map (torch.Tensor): Voltage map input of shape [B, 1, H, W].
s_1d (torch.Tensor): Sequential input of shape [B, C, T].
p_feat (torch.Tensor): Static point features of shape [B, F].
return_attention (bool): If True, also return sequence attention weights.
Returns:
torch.Tensor or tuple:
Predicted normalized SOH, or a tuple of
(prediction, attention_weights) when return_attention=True.
"""
# --- 1. Spatial branch: Vision Mamba processing ---
vim_out = self.vim(v_map)
vim_out = self.norm_vim(vim_out)
# --- 2. Temporal branch: Mamba over 1D sequential signals ---
s_1d = s_1d.transpose(1, 2)
batch_size, seq_len, _ = s_1d.shape
# Append normalized temporal position to preserve sequence order information
time_pos = torch.linspace(0, 1, steps=seq_len, device=s_1d.device)
time_pos = time_pos.view(1, seq_len, 1).expand(batch_size, seq_len, 1)
s_1d = torch.cat([s_1d, time_pos], dim=2)
m_out = self.seq_proj(s_1d)
m_out = self.seq_mamba(m_out)
# Attention pooling converts the temporal sequence into a fixed-size vector
attn_weights = self.seq_attention(m_out)
seq_out = torch.sum(m_out * attn_weights, dim=1)
seq_out = self.norm_seq(seq_out)
# --- 3. Static branch: point-feature processing ---
p_encoded = self.point_encoder(p_feat)
p_encoded = self.norm_point(p_encoded)
# --- 4. Multimodal fusion and regression ---
combined = torch.cat([vim_out, seq_out, p_encoded], dim=1)
combined = self.fusion_norm(combined)
if self.training:
# Light dropout regularizes the fused representation during training
combined = nn.functional.dropout(combined, p=0.02)
out = self.regressor(combined)
if return_attention:
return out, attn_weights
return out
# ==============================================================================
# 4. EVALUATE ALL 25 MODELS (5 Seeds × 5 Runs)
# ==============================================================================
model_seeds = [44, 400, 300, 30, 20]
run_suffixes = ["a", "b", "c", "d", "e"]
# Per-seed summary metrics
seed_mapes = []
104
seed_rmses = []
seed_max_errs = []
# Per-run metrics across all 25 runs
all_run_mapes = []
all_run_rmses = []
all_run_maxerr = []
# Track the best single run for downstream visualization
global_best_mape = float("inf")
best_run_preds = None
best_run_lbls = None
best_test_features = None
best_model_state = None
best_dataset_params = None
run_seed_tracker = [] # Used to build the boxplot of run-wise MAPE
print(f"\n Evaluating 25 models (5 seeds × 5 runs)...\n")
for seed in model_seeds:
print(f"\n{’=’*60}")
print(f" Evaluating SEED {seed}")
print(f"{’=’*60}")
# Reconstruct the deterministic split associated with this seed
set_seed(seed)
features_for_this_seed = list(clean_features)
random.shuffle(features_for_this_seed)
split_idx = int(0.8 * len(features_for_this_seed))
train_features = features_for_this_seed[:split_idx]
test_features = features_for_this_seed[split_idx:]
# Recompute point-feature normalization from the train split only
train_points = np.stack(
[np.nan_to_num(f["point_features"], nan=0.0) for f in train_features]
)
point_mean = torch.FloatTensor(train_points.mean(axis=0))
point_std = torch.FloatTensor(train_points.std(axis=0))
point_std[point_std == 0] = 1.0
test_dataset = OptimalBatteryDataset(test_features, point_mean, point_std)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
this_seed_mapes = []
this_seed_rmses = []
this_seed_max_errs = []
for idx, suffix in enumerate(run_suffixes):
model_path = f"{DRIVE_PREFIX}/Mamba2/best_Unified_{seed}_{suffix}.pth"
print(f" Loading Run ’{suffix.upper()}’: {model_path}")
model = UnifiedMambaSOH().to(device)
try:
model.load_state_dict(
torch.load(model_path, map_location=device, weights_only=True)
)
except Exception as e:
105
print(f" Could not load {model_path}: {e}. Skipping.")
continue
model.eval()
run_preds = []
run_lbls = []
val_mape_sum = 0.0
valid_val_batches = 0
# Evaluation mode disables training-time behaviours and gradient tracking
with torch.no_grad():
for v_map, s_1d, p_feat, lbl in test_loader:
v_map, s_1d, p_feat, lbl = (
v_map.to(device),
s_1d.to(device),
p_feat.to(device),
lbl.to(device),
)
lbl = lbl.view(-1, 1)
out_norm = model(v_map, s_1d, p_feat).view(-1, 1)
# Convert normalized outputs and labels back to capacity in Ah
out_real = out_norm * 155.0
lbl_real = lbl * 155.0
# MAPE is computed as the sum of sample-wise absolute percentage errors
abs_pct = (
torch.sum(torch.abs(out_real - lbl_real) / (lbl_real + 1e-6)) * 100.0
)
if not torch.isnan(abs_pct):
val_mape_sum += abs_pct.item()
valid_val_batches += lbl.size(0)
run_preds.append(out_real.cpu().numpy())
run_lbls.append(lbl_real.cpu().numpy())
# Per-run metrics
run_mape = val_mape_sum / valid_val_batches if valid_val_batches > 0 else 999.0
run_preds_arr = np.vstack(run_preds)
run_lbls_arr = np.vstack(run_lbls)
abs_errors = np.abs(run_preds_arr - run_lbls_arr)
# RMSE reported as a percentage of nominal capacity
run_rmse_pct = (np.sqrt(np.mean(abs_errors**2)) / 155.0) * 100.0
# Maximum percentage error across all samples in this run
run_max_err = np.max((abs_errors / (run_lbls_arr + 1e-6)) * 100.0)
# Track the globally best run for visualization and interpretability
if run_mape < global_best_mape:
global_best_mape = run_mape
best_run_preds = run_preds_arr.flatten()
best_run_lbls = run_lbls_arr.flatten()
best_test_features = test_features
best_model_state = model.state_dict()
106
best_dataset_params = (point_mean, point_std)
this_seed_mapes.append(run_mape)
this_seed_rmses.append(run_rmse_pct)
this_seed_max_errs.append(run_max_err)
all_run_mapes.append(run_mape)
all_run_rmses.append(run_rmse_pct)
all_run_maxerr.append(run_max_err)
run_seed_tracker.append(
{"Seed": str(seed), "MAPE": run_mape, "RMSE": run_rmse_pct}
)
print(
f" Run {suffix.upper()} → MAPE: {run_mape:.2f}% | "
f"RMSE: {run_rmse_pct:.2f}% | Max Error: {run_max_err:.2f}%"
)
# Per-seed averages over the five runs
seed_avg_mape = np.mean(this_seed_mapes)
seed_avg_rmse = np.mean(this_seed_rmses)
seed_avg_max = np.mean(this_seed_max_errs)
seed_mapes.append(seed_avg_mape)
seed_rmses.append(seed_avg_rmse)
seed_max_errs.append(seed_avg_max)
print(
f"\n Seed {seed} Average → MAPE: {seed_avg_mape:.2f}% | "
f"RMSE: {seed_avg_rmse:.2f}% | Max Error: {seed_avg_max:.2f}%"
)
# ==============================================================================
# 5. AGGREGATE RESULTS AND BUILD THE ACADEMIC TABLE
# ==============================================================================
# Overall mean and standard deviation across all 25 runs
avg_mape = np.mean(all_run_mapes)
std_mape = np.std(all_run_mapes)
avg_rmse = np.mean(all_run_rmses)
std_rmse = np.std(all_run_rmses)
avg_max = np.mean(all_run_maxerr)
std_max = np.std(all_run_maxerr)
best_mape = np.min(all_run_mapes)
print(f"\n\n{’-’*60}")
print(" FINAL RESULTS ACROSS ALL 25 RUNS")
print(f"{’-’*60}")
print(f" Best Single Run MAPE : {best_mape:.2f}%")
print(f" Average MAPE : {avg_mape:.2f}% ± {std_mape:.3f}%")
print(f" Average RMSE : {avg_rmse:.2f}% ± {std_rmse:.3f}%")
print(f" Average Max Error : {avg_max:.2f}% ± {std_max:.3f}%")
# Build a per-seed table for academic reporting
table_data = []
for i, seed in enumerate(model_seeds):
table_data.append(
107
{
("Seed", ""): str(seed),
("MAPE (%)", "Average"): f"{seed_mapes[i]:.2f}",
("MAPE (%)", "Std."): f"{np.std([all_run_mapes[i * 5 + j] for j in range(5)]):.3f}",
("RMSE (%)", "Average"): f"{seed_rmses[i]:.2f}",
("RMSE (%)", "Std."): f"{np.std([all_run_rmses[i * 5 + j] for j in range(5)]):.3f}",
("Max Error (%)", "Avg"): f"{seed_max_errs[i]:.2f}",
}
)
# Append a final overall summary row
table_data.append(
{
("Seed", ""): "Overall",
("MAPE (%)", "Average"): f"{avg_mape:.2f}",
("MAPE (%)", "Std."): f"{std_mape:.3f}",
("RMSE (%)", "Average"): f"{avg_rmse:.2f}",
("RMSE (%)", "Std."): f"{std_rmse:.3f}",
("Max Error (%)", "Avg"): f"{avg_max:.2f}",
}
)
df_results = pd.DataFrame(table_data)
df_results.columns = pd.MultiIndex.from_tuples(df_results.columns)
try:
# Left-align headers and values for notebook-friendly academic presentation
styled_df = df_results.style.set_properties(**{"text-align": "left"}).set_table_styles(
[{"selector": "th", "props": [("text-align", "left")]}]
)
display(styled_df)
except NameError:
print(df_results)
# ==============================================================================
# 6. GENERATE ACADEMIC PLOTS
# ==============================================================================
print("\n Generating Academic Plots...")
# Use the best single run for all diagnostic visualizations
preds_cap = best_run_preds
actuals_cap = best_run_lbls
# Convert capacity back to normalized SOH
preds_soh = preds_cap / 155.0
actuals_soh = actuals_cap / 155.0
# ---------------------------------------------------------
# Plot 1a: Predicted vs Actual SOH
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=actuals_soh, y=preds_soh, alpha=0.5, edgecolor=None, color="#1f77b4")
# Dynamic axis limits with a small margin to avoid clipping
min_soh = min(actuals_soh.min(), preds_soh.min()) - 0.02
max_soh = max(actuals_soh.max(), preds_soh.max()) + 0.02
108
plt.xlim(min_soh, max_soh)
plt.ylim(min_soh, max_soh)
plt.plot([min_soh, max_soh], [min_soh, max_soh], "r--", lw=2, label="Ideal Fit (y=x)")
plt.xlabel("Measured SOH", fontweight="bold")
plt.ylabel("Predicted SOH", fontweight="bold")
plt.title("Best Run: Predicted vs. Actual SOH", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# Plot 1b: Predicted vs Actual Capacity
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.scatterplot(x=actuals_cap, y=preds_cap, alpha=0.5, edgecolor=None, color="#2ca02c")
# Dynamic axis limits with a 2 Ah buffer
min_cap = min(actuals_cap.min(), preds_cap.min()) - 2.0
max_cap = max(actuals_cap.max(), preds_cap.max()) + 2.0
plt.xlim(min_cap, max_cap)
plt.ylim(min_cap, max_cap)
plt.plot([min_cap, max_cap], [min_cap, max_cap], "r--", lw=2, label="Ideal Fit (y=x)")
plt.xlabel("Measured Capacity (Ah)", fontweight="bold")
plt.ylabel("Predicted Capacity (Ah)", fontweight="bold")
plt.title("Best Run: Predicted vs. Actual Capacity", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# Plot 2: Residual Error Distribution
# ---------------------------------------------------------
residuals_cap = preds_cap - actuals_cap
plt.figure(figsize=(8, 6))
sns.histplot(residuals_cap, kde=True, bins=40, color="#ff7f0e")
plt.axvline(x=0, color="red", linestyle="--", lw=2)
plt.xlabel("Residual Error (Predicted - Actual Capacity in Ah)", fontweight="bold")
plt.ylabel("Frequency", fontweight="bold")
plt.title("Distribution of Residual Errors (Capacity)", fontweight="bold")
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# Plot 3: Boxplots of MAPE Across Seeds
# ---------------------------------------------------------
df_metrics = pd.DataFrame(run_seed_tracker)
plt.figure(figsize=(10, 6))
sns.boxplot(x="Seed", y="MAPE", data=df_metrics, palette="Set2")
sns.stripplot(x="Seed", y="MAPE", data=df_metrics, color=".25", size=6, jitter=True)
plt.xlabel("Random Seed", fontweight="bold")
plt.ylabel("Test MAPE (%)", fontweight="bold")
plt.title("Model Stability: MAPE Distribution Across 25 Runs", fontweight="bold")
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# Plot 4: Capacity Degradation Trajectory for Selected Vehicles
109
# ---------------------------------------------------------
df_traj = pd.DataFrame(
{
"vehicle_id": [f["vehicle_id"] for f in best_test_features],
"session_id": [f["session_id"] for f in best_test_features],
"actual_cap": actuals_cap,
"pred_cap": preds_cap,
}
)
# Select vehicles with the longest test-set histories for illustrative plotting
top_vehicles = df_traj["vehicle_id"].value_counts().head(2).index.tolist()
if top_vehicles:
plt.figure(figsize=(12, 6))
colors = ["#1f77b4", "#d62728"]
global_min_traj = float("inf")
global_max_traj = float("-inf")
for i, vid in enumerate(top_vehicles):
v_data = df_traj[df_traj["vehicle_id"] == vid].sort_values("session_id")
plt.plot(
v_data["session_id"],
v_data["actual_cap"],
label=f"Vehicle {vid} (Actual)",
color=colors[i],
linestyle="-",
marker="o",
markersize=4,
)
plt.plot(
v_data["session_id"],
v_data["pred_cap"],
label=f"Vehicle {vid} (Predicted)",
color=colors[i],
linestyle="--",
alpha=0.7,
)
# Track global limits to scale the y-axis dynamically
local_min = min(v_data["actual_cap"].min(), v_data["pred_cap"].min())
local_max = max(v_data["actual_cap"].max(), v_data["pred_cap"].max())
global_min_traj = min(global_min_traj, local_min)
global_max_traj = max(global_max_traj, local_max)
plt.ylim(global_min_traj - 2.0, global_max_traj + 2.0)
plt.xlabel("Session ID (Cycles/Time)", fontweight="bold")
plt.ylabel("Capacity (Ah)", fontweight="bold")
plt.title("Capacity Degradation Trajectory Tracking", fontweight="bold")
plt.legend()
plt.tight_layout()
plt.show()
# ---------------------------------------------------------
# Plot 5: Sequence Attention Weights
# ---------------------------------------------------------
110
if best_model_state is not None:
# Reconstruct the best-performing model for interpretability analysis
best_model = UnifiedMambaSOH().to(device)
best_model.load_state_dict(best_model_state)
best_model.eval()
# Rebuild the best test dataset using stored train-set normalization parameters
p_mean, p_std = best_dataset_params
best_dataset = OptimalBatteryDataset(best_test_features, p_mean, p_std)
# Use the first test sample as an illustrative example
v_map, s_1d, p_feat, lbl = best_dataset[0]
v_map = v_map.unsqueeze(0).to(device)
s_1d = s_1d.unsqueeze(0).to(device)
p_feat = p_feat.unsqueeze(0).to(device)
with torch.no_grad():
_, attn_weights = best_model(v_map, s_1d, p_feat, return_attention=True)
# Attention weights have shape [1, Seq_Len, 1]; squeeze to 1D for plotting
attn_weights = attn_weights.squeeze().cpu().numpy()
plt.figure(figsize=(12, 4))
plt.plot(attn_weights, color="#2ca02c", lw=2)
plt.fill_between(range(len(attn_weights)), attn_weights, color="#2ca02c", alpha=0.3)
# Dynamic y-axis scaling is used because attention magnitudes may be very small
plt.ylim(0, max(attn_weights) * 1.1)
plt.xlabel("Sequence Step (Time)", fontweight="bold")
plt.ylabel("Attention Weight", fontweight="bold")
plt.title("Mamba Temporal Attention Mechanism (Sample Interpretability)", fontweight="bold")
plt.tight_layout()
plt.show()
print(" All dynamically scaled plots generated successfully!")